(ch:pca)=
# 주성분 분석(PCA)

:::{note} 감사의 글

오렐리앙 제롱<font size='2'>Aurélien Géron</font>의 [Hands-On Machine Learning with Scikit-Learn and PyTorch (O'Reilly, 2025)](https://github.com/ageron/handson-mlp)에 사용된 코드를 참고한 강의노트이다. 보다 심화된 이해를 위해 책 원본을 읽을 것을 강력하게 권장한다. 자료를 공개한 오렐리앙 제롱과 일부 이미지 자료를 제공해 준 한빛아카데미에게 진심어린 감사를 전한다.
:::

:::{seealso} 코드 실행
[(코드 워크아웃) 주성분 분석(PCA)](https://colab.research.google.com/github/codingalzi/code-workout-ml/blob/master/notebooks/code-ensemble_learning.ipynb)을 병행하여 읽을 것을 권장한다.
:::

특성 수가 많은 데이터는 학습이 느려지거나 어려워질 수 있다.
예를 들어 온라인 쇼핑몰 고객 데이터를 클릭한 상품 종류, 상품별 체류 시간, 검색어, 접속 시간대, 사용 기기, 쿠폰 사용 여부 등 수많은 특성으로 표현하면 고객들을 서로 비교하기가 어려워진다.
이런 상황에서는 새 고객과 비슷한 기존 고객을 찾기 어렵고, 모델이 일반적인 구매 패턴보다 훈련셋에만 우연히 나타난 특징에 지나치게 맞춰질 위험도 커진다.

**PCA**(Principal Component Analysis), 즉 **주성분 분석**은 이런 고차원 데이터를 더 적은 수의 새 특성으로 압축하는 대표적인 차원 축소 기법이다.
PCA의 목표는 정보 손실을 어느 정도 감수하더라도 데이터의 중요한 구조를 최대한 유지하면서 훈련 속도와 성능을 좋게 만드는 것이다.

## 차원의 저주

하나의 샘플은 여러 특성값으로 표현된다.
예를 들어 온라인 쇼핑몰 고객 한 명을 `최근 구매 금액`, `방문 횟수` 두 가지 특성으로만 표현하면 각 고객은 2차원 공간의 한 점으로 생각할 수 있다.
여기에 `클릭한 상품 종류`, `상품별 체류 시간`, `검색어`, `접속 시간대`, `사용 기기`, `쿠폰 사용 여부` 같은 특성을 계속 추가하면 고객 한 명은 더 높은 차원의 공간에 놓인 점이 된다.
이처럼 데이터셋의 특성 수가 곧 데이터가 놓이는 공간의 차원이다.

특성이 많아지면 데이터를 더 자세히 설명할 수 있다는 장점이 있다.
하지만 특성 수가 지나치게 많아지면 같은 개수의 샘플이 훨씬 넓은 공간에 흩어져 있는 것처럼 된다.
2차원 지도에서는 가까운 고객을 찾기 쉽지만, 수십 개나 수백 개의 기준을 동시에 고려하면 어떤 고객들이 서로 정말 비슷한지 판단하기가 어려워진다.
새 고객 데이터가 들어왔을 때도 마찬가지다.
훈련셋 안에서 이 고객과 충분히 비슷한 기존 고객을 찾기 어려우면, 기존 고객의 구매 패턴을 바탕으로 새 고객의 행동을 추정하기도 어려워진다.

이 상황에서는 과대적합 위험도 커진다.
모델이 전체 고객에게 공통으로 나타나는 일반적인 패턴을 배우기보다, 훈련셋에 우연히 포함된 세부적인 조합에 지나치게 맞춰질 수 있기 때문이다.
예를 들어 어떤 훈련 샘플 몇 개에서만 나타난 `특정 시간대`, `특정 기기`, `특정 검색어`, `특정 쿠폰 사용 여부`의 조합을 모델이 중요한 규칙처럼 기억해버릴 수 있다.
훈련셋에서는 잘 맞는 것처럼 보이지만, 새로운 고객 데이터에는 잘 일반화되지 않을 가능성이 높다.
이처럼 특성 수가 너무 많아 학습이 느려지거나 어려워지고, 과대적합 위험까지 커지는 현상을 **차원의 저주**라 부른다.

**차원 축소**

차원의 저주를 완화하는 가장 직접적인 방법은 훈련 샘플을 훨씬 많이 모으는 것이다.
샘플이 충분히 많아지면 넓은 고차원 공간도 더 촘촘하게 채울 수 있기 때문이다.
하지만 차원이 늘어날수록 필요한 샘플 수는 매우 빠르게 증가한다.
현실의 데이터 분석에서는 과대적합을 피할 만큼 많은 샘플을 모으기가 어렵거나, 비용과 시간이 너무 많이 들 수 있다.
그래서 많은 경우에는 모든 특성을 그대로 사용하기보다, 데이터의 중요한 정보를 최대한 유지하면서 특성 수를 줄이는 방법을 고려한다.
이것이 **차원 축소**다.

차원 축소의 핵심은 정보를 무작정 버리는 것이 아니다.
서로 비슷한 정보를 담고 있는 특성들을 압축하거나, 데이터의 중요한 변화가 잘 드러나는 몇 개의 새로운 기준으로 데이터를 다시 표현하는 것이다.
예를 들어 고객 데이터에서 여러 클릭 기록과 체류 시간 특성들이 모두 `전자제품 관심도`와 관련되어 있다면, 이 많은 특성을 하나의 핵심적인 행동 특성으로 요약할 수 있다.
또 여러 할인 쿠폰 관련 특성이 `가격 민감도`를 함께 설명한다면, 이 역시 더 적은 수의 특성으로 압축할 수 있다.
이렇게 특성 수를 줄이면 모델이 다루어야 할 공간이 작아지고, 훈련 속도가 빨라지며, 불필요한 세부 정보에 맞춰지는 위험도 줄어들 수 있다.

**주성분 분석(PCA)**

차원 축소를 수행하는 가장 널리 사용되는 방법 중 하나가 주성분 분석(PCA) 기법이다.
PCA는 원래 특성 중 일부를 직접 고르는 방법이 아니다.
대신 원래 특성들을 조합하여 데이터가 가장 많이 퍼져 있는 방향을 찾고, 그 방향들을 새로운 축으로 사용한다.
이 새로운 축을 **주성분**이라 부른다.
데이터가 많이 퍼져 있는 방향은 샘플들 사이의 차이를 잘 드러내는 방향이므로, 그 방향을 보존하면 원래 데이터의 중요한 구조를 비교적 잘 유지할 수 있다.

PCA는 모든 주성분을 다 사용하지 않고, 중요한 주성분 몇 개만 선택하여 데이터를 더 낮은 차원으로 표현한다.
예를 들어 784개의 픽셀값으로 표현된 MNIST 손글씨 이미지를 154개의 주성분으로 줄이면, 원래보다 훨씬 적은 차원으로도 숫자를 구분하는 데 필요한 정보를 상당 부분 유지할 수 있다.
즉 PCA는 고차원 데이터를 더 다루기 쉬운 저차원 표현으로 바꾸면서, 원래 데이터의 중요한 차이를 최대한 보존하려는 기법이다.
따라서 전체 흐름은 다음과 같이 정리할 수 있다.

```text
차원의 저주 -> 차원 축소 필요 -> PCA 활용
```

단, PCA가 차원의 저주를 항상 완전히 해결하는 것은 아니다.
PCA는 여러 특성 사이에 중복되거나 관련된 정보가 많고, 데이터의 중요한 변화가 비교적 적은 수의 방향에 잘 모여 있을 때 특히 효과적이다.
반대로 중요한 정보가 분산이 작은 방향에 숨어 있거나, 데이터의 구조가 단순한 직선적 방향으로 잘 설명되지 않는 경우에는 PCA만으로 충분하지 않을 수 있다.
따라서 PCA는 차원의 저주를 완화하기 위한 매우 유용한 출발점이지만, 데이터의 성격과 분석 목적에 맞게 사용해야 한다.

## PCA의 기본 아이디어

PCA는 데이터를 가장 잘 설명하는 방향을 차례대로 찾는다.
데이터가 가장 넓게 퍼져 있는 방향을 첫째 주성분으로 잡고, 그 다음으로 많이 퍼져 있는 방향을 둘째 주성분으로 잡는 식이다.
이렇게 찾은 주성분들을 새로운 좌표축으로 사용하면 원래 데이터의 중요한 변화를 비교적 적은 차원으로 표현할 수 있다.

### 사영과 초평면

PCA는 훈련 데이터셋을 특정 초평면<font size='2'>hyperplane</font>에 사영하는 방식으로 차원을 줄인다.
이때 초평면은 주성분 분석을 통해 결정된다.
초평면 지정에 사용되는 **주성분**은 **분산 보존** 개념과 밀접하게 연관된다.

:::{note} 초평면

초평면<font size='2'>hyperplane</font>은 고차원 공간에서 직선이나 평면의 역할을 일반화한 개념이다.
아래 방정식을 만족하는 벡터들의 집합으로 표현할 수 있다.

$$
a_1 x_1 + a_2 x_2 + \cdots + a_n x_n + c = 0
$$

위 식을 차원별로 보면 다음과 같다.

- `n=1`인 경우: 1차원 공간의 점
- `n=2`인 경우: 2차원 공간의 직선
- `n=3`인 경우: 3차원 공간의 평면
- `n>=4`인 경우: n차원 공간의 초평면
:::

### 분산 보존

저차원으로 사영할 때는 데이터셋의 분산이 최대한 유지되도록 축을 지정해야 한다.
분산이 많이 보존된다는 것은 데이터가 서로 얼마나 다르게 분포하는지에 대한 정보가 많이 남는다는 뜻이다.
아래 그림에서 $c_1$ 벡터가 위치한 실선 축으로 사영하는 경우가 분산을 가장 많이 보존한다.

<div align="center"><img src="https://raw.githubusercontent.com/codingalzi/handson-ml3/master/jupyter-book/imgs/ch08/homl08-09.png" width="500"/></div>

### 주성분

주성분은 다음 과정으로 차례대로 찾는다.

- 첫째 주성분: 분산을 최대한 보존하는 축
- 둘째 주성분: 첫째 주성분과 수직을 이루면서, 첫째 주성분이 담당하지 않는 분산을 최대한 보존하는 축
- 셋째 주성분: 첫째와 둘째 주성분에 수직이면서, 앞의 두 주성분이 담당하지 않는 분산을 최대한 보존하는 축
- 이후 주성분도 같은 방식으로 찾는다.

사영에 사용되는 초평면은 주성분으로 구성된 축을 이용하는 공간으로 지정한다.
예를 들어 첫째와 둘째 주성분만 축으로 사용하면 2차원 초평면이 생성된다.

### 특잇값 분해(SVD)

데이터셋의 주성분은 선형대수의 **특잇값 분해**<font size='2'>Singular Value Decomposition</font>(SVD) 기법을 이용하여 효율적으로 찾을 수 있다.
SVD를 사용하면 주성분을 계산하고, 찾아진 초평면으로 데이터를 사영하는 과정도 비교적 쉽게 처리된다.
단, 데이터셋이 크거나 특성이 많으면 계산 시간이 길어질 수 있다.

:::{important} 스케일 조정
PCA는 분산을 기준으로 주성분을 찾기 때문에 특성의 스케일에 민감하다.
예를 들어 한 특성은 원 단위이고 다른 특성은 0과 1 사이의 비율이라면, 값의 범위가 큰 특성이 주성분 계산에 지나치게 큰 영향을 줄 수 있다.
따라서 일반적인 수치형 데이터에는 PCA를 적용하기 전에 `StandardScaler` 등으로 스케일을 맞추는 것이 좋다.
단, MNIST 픽셀값처럼 모든 특성이 같은 단위와 범위를 갖는 경우에는 상황에 따라 별도의 표준화가 필요하지 않을 수 있다.
:::

## 사이킷런의 `PCA` 모델

사이킷런의 `PCA` 모델은 SVD 기법을 활용한다.
예를 들어 아래 코드는 데이터셋의 차원을 2로 줄인다.

```python
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
X2D = pca.fit_transform(X)
```

### 설명 분산 비율

훈련된 `PCA` 모델의 `explained_variance_ratio_` 속성에는 각 주성분이 원 데이터셋의 분산을 얼마나 설명하는지가 저장된다.
예를 들어 아래 사영 그림에서 3차원 데이터셋을 새로운 축 $z_1$과 $z_2$로 표현하면, 각 축이 차지하는 분산 비율은 다음과 같다.

- $z_1$ 축: 75.8%
- $z_2$ 축: 15.2%

```python
>>> pca.explained_variance_ratio_
array([0.7578477 , 0.15186921])
```

<table>
    <tr>
        <td> <div align="center"><img src="https://raw.githubusercontent.com/codingalzi/handson-ml3/master/jupyter-book/imgs/ch08/homl08-02-1.png" width="400"/></div> </td>
        <td></td>
        <td> <div align="center"><img src="https://raw.githubusercontent.com/codingalzi/handson-ml3/master/jupyter-book/imgs/ch08/homl08-02-2.png" width="400"/></div> </td>
    </tr>
</table>

## 적절한 차원 선택

설명 분산 비율의 합이 95% 정도 되도록 주성분의 개수를 정하는 방법이 자주 사용된다.
반면 데이터 시각화가 목적이라면 사람이 눈으로 확인할 수 있도록 2개 또는 3개의 주성분만 사용해야 한다.

### 설명 분산 비율 활용

적절한 차원을 결정하기 위해 설명 분산 비율의 누적합과 차원 사이의 그래프를 활용할 수 있다.
예를 들어 설명 분산 비율의 누적합 증가가 완만하게 변하는 지점, 즉 팔꿈치<font size='2'>elbow</font> 지점을 살펴보면 좋다.

<div align="center"><img src="https://raw.githubusercontent.com/codingalzi/handson-ml3/master/jupyter-book/imgs/ch08/homl08-10.png" width="400"/></div>

위 그래프를 통해 설명 분산 비율의 합이 95% 정도가 되려면 154개의 차원이 필요함을 확인할 수 있다.
따라서 `n_components=154`를 하이퍼파라미터로 지정할 수 있으나, 이보다는 `n_components=0.95`로 지정하는 것이 더 편리하다.

```python
pca = PCA(n_components=0.95)
X_reduced = pca.fit_transform(X_train)
```

`n_components` 하이퍼파라미터에 정수를 사용하면 줄일 차원의 수를 직접 지정한다.
반면 0과 1 사이의 부동소수점을 지정하면 보존할 설명 분산 비율을 지정한다.

### 파이프라인과 랜덤 탐색 활용

적절한 차원을 찾기 위해 `PCA`를 전처리로 사용하는 파이프라인을 생성하고 랜덤 탐색을 이용할 수 있다.
예를 들어 아래 코드는 차원 축소와 랜덤 포레스트 모델을 하나의 파이프라인으로 묶은 후, 랜덤 탐색을 이용하여 적절한 주성분 개수를 찾는다.

```python
import numpy as np

from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.pipeline import make_pipeline

clf = make_pipeline(PCA(random_state=42),
                    RandomForestClassifier(random_state=42))

param_distrib = {
    "pca__n_components": np.arange(10, 80),
    "randomforestclassifier__n_estimators": np.arange(50, 500)
}

rnd_search = RandomizedSearchCV(clf, param_distrib, n_iter=10, cv=3,
                                random_state=42)
rnd_search.fit(X_train[:1000], y_train[:1000])
```

## PCA 활용 예제: 데이터 압축

PCA는 데이터 압축 용도로 활용할 수 있다.
MNIST 데이터셋의 경우 784차원을 154차원으로 줄이면 데이터셋의 크기가 원래의 20% 수준이 되어 훈련 속도가 훨씬 빨라진다.
하지만 설명 분산 비율을 95% 정도로 유지하면 정보는 5% 정도만 잃는다.
아래 그림은 정보 손실이 크지 않음을 보여준다.
왼쪽이 원본이고 오른쪽이 압축된 데이터를 다시 복원한 결과다.

<div align="center"><img src="https://raw.githubusercontent.com/codingalzi/handson-ml3/master/jupyter-book/imgs/ch08/homl08-11.png" width="400"/></div>

:::{note} 특성 선택과 PCA의 차이
PCA는 원래 특성 중 일부만 고르는 방법이 아니다.
여러 원래 특성의 정보를 조합하여 **주성분**이라는 새로운 특성을 만든다.
따라서 MNIST 데이터셋에서 784개 픽셀 정보를 154개의 주성분으로 줄인다는 말은 픽셀 154개만 선택한다는 뜻이 아니라, 784차원 정보를 154차원 표현으로 압축한다는 뜻이다.
:::

## 랜덤 PCA

랜덤 PCA는 주성분 선택에 사용되는 SVD 알고리즘을 확률적으로 작동하도록 만든 기법이다.
지정된 개수의 주성분에 대한 근삿값을 더 빠르게 찾아준다.

```python
rnd_pca = PCA(n_components=154, svd_solver="randomized")
X_reduced = rnd_pca.fit_transform(X_train)
```

## 점진적 PCA

점진적 PCA, 즉 IPCA<font size='2'>Incremental PCA</font>는 훈련 세트를 미니배치로 나눈 후 하나씩 주입하여 학습하는 모델이다.
전체 훈련 세트를 한 번에 메모리에 올리기 어려운 경우나 온라인 학습 상황에 활용할 수 있다.
점진적 PCA는 훈련에 `partial_fit()`을 사용한다.

```python
import numpy as np

from sklearn.decomposition import IncrementalPCA

n_batches = 100
inc_pca = IncrementalPCA(n_components=154)

for X_batch in np.array_split(X_train, n_batches):
    inc_pca.partial_fit(X_batch)

X_reduced = inc_pca.transform(X_train)
```

### `memmap` 클래스 활용

넘파이의 `memmap` 클래스는 바이너리 파일로 저장된 매우 큰 데이터셋을 마치 메모리에 들어 있는 배열처럼 다룰 수 있게 해준다.
이를 이용하면 큰 데이터셋에 대해서도 미니배치 방식의 점진적 PCA를 적용할 수 있다.

```python
import numpy as np

from sklearn.decomposition import IncrementalPCA

n_batches = 100

# memmap 생성
filename = "my_mnist.mmap"
X_mmap = np.memmap(filename, dtype="float32", mode="w+", shape=X_train.shape)
X_mmap[:] = X_train
X_mmap.flush()

# memmap 활용
X_mmap = np.memmap(filename, dtype="float32", mode="r").reshape(-1, 784)

batch_size = X_mmap.shape[0] // n_batches
inc_pca = IncrementalPCA(n_components=154, batch_size=batch_size)
inc_pca.fit(X_mmap)
```

## 요약

- PCA는 고차원 데이터를 더 적은 수의 주성분으로 압축하는 대표적인 차원 축소 기법이다.
- PCA는 원래 특성 일부를 선택하는 것이 아니라, 원래 특성들의 조합으로 새로운 축을 만든다.
- 주성분은 데이터의 분산을 최대한 보존하는 방향으로 차례대로 선택된다.
- `explained_variance_ratio_`와 `n_components`를 이용하면 보존할 정보량과 축소할 차원을 조절할 수 있다.
- 큰 데이터셋에는 랜덤 PCA나 점진적 PCA를 활용할 수 있다.